# Pipeline 4: Bad Exit Early Warning

## 1. Problem Framing

**Business Question:** Which factors predict that a young woman will have a bad exit from the lighthouse?

**Stakeholder:** Case managers who need to identify at-risk residents *early* so they can intensify interventions before negative outcomes occur.

**Why it matters:** The organization's greatest fear is girls falling through the cracks. An early warning system can flag residents at risk of transfer, runaway, or regression while there is still time to intervene.

**Target Variable:** Binary -- "Bad exit" defined as:
- `case_status == 'Transferred'`, OR
- Multiple runaway attempts or self-harm incidents, OR
- `reintegration_status == 'On Hold'` with declining indicators

**This pipeline is distinct from Pipeline 3** because it focuses on *early-stage warning* using intake-time and early-trajectory features, not full-lifecycle outcome prediction.

**Approach:**
- **Explanatory model (Logistic Regression):** Identify early risk factors.
- **Predictive model (Gradient Boosting + Decision Tree):** Maximize recall to catch at-risk residents even at the cost of some false positives.

**Success metric:** Recall (sensitivity) is the primary metric. Missing a girl at risk is the worst outcome.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, recall_score, f1_score,
                             precision_recall_curve)
from sklearn.feature_selection import RFE
import statsmodels.api as sm
import joblib, os
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
DATA_DIR = '../Lighthouse_Project_CSVs'

In [2]:
residents = pd.read_csv(f'{DATA_DIR}/residents.csv', parse_dates=['date_of_birth','date_of_admission','date_closed'])
process_rec = pd.read_csv(f'{DATA_DIR}/process_recordings.csv', parse_dates=['session_date'])
education = pd.read_csv(f'{DATA_DIR}/education_records.csv', parse_dates=['record_date'])
health = pd.read_csv(f'{DATA_DIR}/health_wellbeing_records.csv', parse_dates=['record_date'])
interventions = pd.read_csv(f'{DATA_DIR}/intervention_plans.csv')
incidents = pd.read_csv(f'{DATA_DIR}/incident_reports.csv', parse_dates=['incident_date'])
home_visits = pd.read_csv(f'{DATA_DIR}/home_visitations.csv', parse_dates=['visit_date'])

# Define bad exit target
residents['bad_exit'] = (
    (residents['case_status'] == 'Transferred') |
    (residents['reintegration_status'] == 'On Hold')
).astype(int)

# Add high-incident residents
inc_counts = incidents.groupby('resident_id').apply(
    lambda g: ((g['incident_type']=='RunawayAttempt').sum() >= 2) | ((g['incident_type']=='SelfHarm').sum() >= 2)
).reset_index(name='high_incidents')
residents = residents.merge(inc_counts, on='resident_id', how='left')
residents['high_incidents'] = residents['high_incidents'].fillna(False).astype(int)
residents['bad_exit'] = (residents['bad_exit'] | residents['high_incidents']).astype(int)

print(f'Bad exit count: {residents.bad_exit.sum()} / {len(residents)}')
print(f'Bad exit rate: {residents.bad_exit.mean():.1%}')

Bad exit count: 29 / 60
Bad exit rate: 48.3%


## 2. Data Acquisition, Preparation & Exploration

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

residents['bad_exit'].value_counts().plot.bar(ax=axes[0,0], color=['forestgreen','salmon'])
axes[0,0].set_title('Target: Bad Exit vs Not')
axes[0,0].set_xticklabels(['Good Outcome (0)', 'Bad Exit (1)'], rotation=0)

# Bad exit by case category
ct = pd.crosstab(residents['case_category'], residents['bad_exit'])
ct.plot.bar(stacked=True, ax=axes[0,1], color=['forestgreen','salmon'])
axes[0,1].set_title('Bad Exit by Case Category')

# Bad exit by initial risk
risk_order = ['Low', 'Medium', 'High', 'Critical']
ct2 = pd.crosstab(residents['initial_risk_level'], residents['bad_exit']).reindex(risk_order)
ct2.plot.bar(stacked=True, ax=axes[1,0], color=['forestgreen','salmon'])
axes[1,0].set_title('Bad Exit by Initial Risk Level')

# Incident types for bad vs non-bad
bad_ids = set(residents[residents['bad_exit']==1]['resident_id'])
inc_bad = incidents[incidents['resident_id'].isin(bad_ids)]['incident_type'].value_counts()
inc_good = incidents[~incidents['resident_id'].isin(bad_ids)]['incident_type'].value_counts()
comp = pd.DataFrame({'Bad Exit': inc_bad, 'Good Outcome': inc_good}).fillna(0)
comp.plot.bar(ax=axes[1,1], color=['salmon','forestgreen'])
axes[1,1].set_title('Incident Types: Bad Exit vs Good')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Feature Engineering -- Emphasis on Early Indicators

We focus on features available early in the resident's stay to make predictions actionable for early intervention.

In [4]:
# --- Intake-time features from residents table ---
risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}

res_features = residents[['resident_id', 'bad_exit']].copy()
res_features['age_at_admission'] = (
    (residents['date_of_admission'] - residents['date_of_birth']).dt.days / 365.25
)
res_features['initial_risk'] = residents['initial_risk_level'].map(risk_map)

sub_cats = ['sub_cat_orphaned','sub_cat_trafficked','sub_cat_child_labor','sub_cat_physical_abuse',
            'sub_cat_sexual_abuse','sub_cat_osaec','sub_cat_cicl','sub_cat_at_risk','sub_cat_street_child']
for col in sub_cats:
    res_features[col] = residents[col].astype(int)
res_features['compound_trauma_score'] = res_features[sub_cats].sum(axis=1)

res_features['is_pwd'] = residents['is_pwd'].astype(int)
res_features['has_special_needs'] = residents['has_special_needs'].astype(int)
res_features['family_is_4ps'] = residents['family_is_4ps'].astype(int)
res_features['family_solo_parent'] = residents['family_solo_parent'].astype(int)
res_features['family_indigenous'] = residents['family_indigenous'].astype(int)
res_features['family_informal_settler'] = residents['family_informal_settler'].astype(int)

# Referral source encoding
ref_map = {'Self-Referral': 0, 'Community': 1, 'NGO': 2, 'Government Agency': 3, 'Police': 4, 'Court Order': 5}
res_features['referral_severity'] = residents['referral_source'].map(ref_map).fillna(0)
print(f'Intake features: {res_features.shape}')

Intake features: (60, 21)


In [5]:
# --- Early counseling features (first 6 months only) ---
emotion_order = {'Distressed': 0, 'Angry': 1, 'Anxious': 2, 'Sad': 3, 'Withdrawn': 4, 'Calm': 5, 'Hopeful': 6, 'Happy': 7}
process_rec['emo_start'] = process_rec['emotional_state_observed'].map(emotion_order)
process_rec['emo_end'] = process_rec['emotional_state_end'].map(emotion_order)
process_rec['emo_delta'] = process_rec['emo_end'] - process_rec['emo_start']

# Join admission date to filter early sessions
pr_with_admit = process_rec.merge(
    residents[['resident_id', 'date_of_admission']], on='resident_id'
)
pr_with_admit['days_since_admission'] = (
    pr_with_admit['session_date'] - pr_with_admit['date_of_admission']
).dt.days
early_sessions = pr_with_admit[pr_with_admit['days_since_admission'] <= 180]

early_counsel = early_sessions.groupby('resident_id').agg(
    early_sessions_count=('recording_id', 'count'),
    early_avg_emo_delta=('emo_delta', 'mean'),
    early_pct_concerns=('concerns_flagged', 'mean'),
    early_pct_progress=('progress_noted', 'mean'),
    early_avg_emo_start=('emo_start', 'mean'),
).reset_index()

# --- Early incident features ---
inc_with_admit = incidents.merge(
    residents[['resident_id', 'date_of_admission']], on='resident_id'
)
inc_with_admit['days_since_admission'] = (
    inc_with_admit['incident_date'] - inc_with_admit['date_of_admission']
).dt.days
early_inc = inc_with_admit[inc_with_admit['days_since_admission'] <= 180]

early_incidents = early_inc.groupby('resident_id').agg(
    early_incident_count=('incident_id', 'count'),
    early_high_severity=('severity', lambda x: (x=='High').sum()),
    early_runaway=('incident_type', lambda x: (x=='RunawayAttempt').sum()),
    early_self_harm=('incident_type', lambda x: (x=='SelfHarm').sum()),
).reset_index()

print(f'Early counseling features: {early_counsel.shape}')
print(f'Early incident features: {early_incidents.shape}')

Early counseling features: (60, 6)


Early incident features: (26, 5)


In [6]:
# --- Early home visit features ---
coop_map = {'Uncooperative': 0, 'Neutral': 1, 'Cooperative': 2, 'Highly Cooperative': 3}
home_visits['coop_score'] = home_visits['family_cooperation_level'].map(coop_map)

hv_with_admit = home_visits.merge(
    residents[['resident_id', 'date_of_admission']], on='resident_id'
)
hv_with_admit['days_since_admission'] = (
    hv_with_admit['visit_date'] - hv_with_admit['date_of_admission']
).dt.days
early_hv = hv_with_admit[hv_with_admit['days_since_admission'] <= 180]

early_visits = early_hv.groupby('resident_id').agg(
    early_visit_count=('visitation_id', 'count'),
    early_pct_unfavorable=('visit_outcome', lambda x: (x=='Unfavorable').mean()),
    early_avg_cooperation=('coop_score', 'mean'),
    early_pct_safety_concerns=('safety_concerns_noted', 'mean'),
).reset_index()

# --- Early health features ---
health_with_admit = health.merge(
    residents[['resident_id', 'date_of_admission']], on='resident_id'
)
health_with_admit['days_since_admission'] = (
    health_with_admit['record_date'] - health_with_admit['date_of_admission']
).dt.days
early_health = health_with_admit[health_with_admit['days_since_admission'] <= 180]

early_health_agg = early_health.groupby('resident_id').agg(
    early_avg_health=('general_health_score', 'mean'),
    early_avg_nutrition=('nutrition_score', 'mean'),
    early_avg_sleep=('sleep_quality_score', 'mean'),
).reset_index()

# --- Intervention stall features ---
int_stall = interventions.groupby('resident_id').agg(
    pct_plans_on_hold=('status', lambda x: (x=='On Hold').mean()),
    pct_plans_open=('status', lambda x: (x=='Open').mean()),
    pct_plans_achieved=('status', lambda x: (x=='Achieved').mean()),
).reset_index()

print(f'Early visit features: {early_visits.shape}')
print(f'Early health features: {early_health_agg.shape}')
print(f'Intervention stall features: {int_stall.shape}')

Early visit features: (57, 5)
Early health features: (60, 4)
Intervention stall features: (60, 4)


In [7]:
# Combine all features
df = res_features \
    .merge(early_counsel, on='resident_id', how='left') \
    .merge(early_incidents, on='resident_id', how='left') \
    .merge(early_visits, on='resident_id', how='left') \
    .merge(early_health_agg, on='resident_id', how='left') \
    .merge(int_stall, on='resident_id', how='left')

num_cols = df.select_dtypes(include=[np.number]).columns.drop(['resident_id', 'bad_exit'])
df[num_cols] = df[num_cols].fillna(0)

print(f'Final feature matrix: {df.shape}')
print(f'Bad exit distribution: {df.bad_exit.value_counts().to_dict()}')
df.head()

Final feature matrix: (60, 40)
Bad exit distribution: {0: 31, 1: 29}


,resident_id,bad_exit,age_at_admission,initial_risk,sub_cat_orphaned,sub_cat_trafficked,sub_cat_child_labor,sub_cat_physical_abuse,sub_cat_sexual_abuse,sub_cat_osaec,...,early_visit_count,early_pct_unfavorable,early_avg_cooperation,early_pct_safety_concerns,early_avg_health,early_avg_nutrition,early_avg_sleep,pct_plans_on_hold,pct_plans_open,pct_plans_achieved
0,1,0,15.126626,3,0,0,0,0,0,0,...,12.0,0.166667,1.083333,0.250000,3.103333,3.210000,3.203333,0.666667,0.000000,0.000000
1,2,0,14.899384,1,0,0,0,0,0,0,...,6.0,0.166667,1.500000,0.333333,3.388571,3.317143,3.334286,0.333333,0.333333,0.333333
2,3,0,17.311431,1,0,0,0,0,1,0,...,10.0,0.300000,0.900000,0.500000,3.058571,2.888571,3.002857,0.333333,0.666667,0.000000
3,4,1,12.246407,2,0,0,0,0,0,0,...,4.0,0.000000,2.500000,0.500000,3.131429,2.940000,2.860000,0.000000,0.333333,0.000000
4,5,1,14.724162,1,0,0,0,1,0,1,...,8.0,0.125000,1.625000,0.250000,3.092857,3.074286,2.961429,0.000000,0.000000,0.333333


In [8]:
# Correlation with bad_exit
target_corr = df.drop(columns=['resident_id'] + sub_cats).corr()['bad_exit'].drop('bad_exit').sort_values()

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['salmon' if x > 0 else 'forestgreen' for x in target_corr]
target_corr.plot.barh(ax=ax, color=colors)
ax.set_title('Feature Correlation with Bad Exit')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Modeling & Feature Selection

### 3a. Explanatory Model: Logistic Regression

In [9]:
feature_candidates = [
    'age_at_admission', 'initial_risk', 'compound_trauma_score',
    'is_pwd', 'has_special_needs', 'family_is_4ps', 'family_solo_parent',
    'family_informal_settler', 'referral_severity',
    'early_sessions_count', 'early_avg_emo_delta', 'early_pct_concerns',
    'early_pct_progress', 'early_avg_emo_start',
    'early_incident_count', 'early_high_severity', 'early_runaway',
    'early_visit_count', 'early_pct_unfavorable', 'early_avg_cooperation',
    'early_pct_safety_concerns',
    'early_avg_health', 'early_avg_nutrition',
    'pct_plans_on_hold', 'pct_plans_achieved',
]
feature_candidates = [f for f in feature_candidates if f in df.columns]

X = df[feature_candidates].fillna(0)
y = df['bad_exit']

# RFE to select top 8
lr_rfe = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
rfe = RFE(lr_rfe, n_features_to_select=8)
rfe.fit(X, y)
selected_features = X.columns[rfe.support_].tolist()
print(f'RFE selected: {selected_features}')

X_sel = X[selected_features]
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_sel), columns=selected_features)

X_sm = sm.add_constant(X_scaled)
logit_model = sm.Logit(y, X_sm).fit(disp=0)
print(logit_model.summary())

odds = np.exp(logit_model.params.drop('const'))
print('\nOdds Ratios (>1 increases bad exit risk):')
print(odds.sort_values(ascending=False))

RFE selected: ['compound_trauma_score', 'is_pwd', 'family_informal_settler', 'early_pct_concerns', 'early_avg_emo_start', 'early_avg_cooperation', 'pct_plans_on_hold', 'pct_plans_achieved']


                           Logit Regression Results                           
Dep. Variable:               bad_exit   No. Observations:                   60
Model:                          Logit   Df Residuals:                       51
Method:                           MLE   Df Model:                            8
Date:                Mon, 06 Apr 2026   Pseudo R-squ.:                  0.1523
Time:                        16:44:42   Log-Likelihood:                -35.227
converged:                       True   LL-Null:                       -41.555
Covariance Type:            nonrobust   LLR p-value:                    0.1242
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.0576      0.289     -0.199      0.842      -0.623       0.508
compound_trauma_score       0.5571      0.315      1.770      0.077      -0.060       1.174


In [10]:
# Odds ratio visualization
fig, ax = plt.subplots(figsize=(10, 6))
odds_sorted = odds.sort_values()
colors = ['salmon' if x > 1 else 'forestgreen' for x in odds_sorted]
odds_sorted.plot.barh(ax=ax, color=colors)
ax.axvline(x=1, color='black', linestyle='--')
ax.set_xlabel('Odds Ratio')
ax.set_title('Bad Exit Odds Ratios')
plt.tight_layout()
plt.show()

### 3b. Predictive Models

In [11]:
pred_features = selected_features.copy()
X_pred = df[pred_features].fillna(0)

cv = StratifiedKFold(5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.1, random_state=42),
}

for name, model in models.items():
    f1 = cross_val_score(model, X_pred, y, cv=cv, scoring='f1')
    recall = cross_val_score(model, X_pred, y, cv=cv, scoring='recall')
    auc = cross_val_score(model, X_pred, y, cv=cv, scoring='roc_auc')
    print(f'{name:25s} | F1: {f1.mean():.3f}+/-{f1.std():.3f} | Recall: {recall.mean():.3f}+/-{recall.std():.3f} | AUC: {auc.mean():.3f}+/-{auc.std():.3f}')

Logistic Regression       | F1: 0.469+/-0.159 | Recall: 0.480+/-0.187 | AUC: 0.564+/-0.167
Decision Tree             | F1: 0.195+/-0.209 | Recall: 0.200+/-0.245 | AUC: 0.403+/-0.101


Random Forest             | F1: 0.461+/-0.242 | Recall: 0.487+/-0.272 | AUC: 0.476+/-0.185


Gradient Boosting         | F1: 0.396+/-0.211 | Recall: 0.420+/-0.238 | AUC: 0.377+/-0.256


In [12]:
# Fit final models
gb_model = GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.1, random_state=42)
gb_model.fit(X_pred, y)

dt_model = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
dt_model.fit(X_pred, y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, class_weight='balanced', random_state=42)
rf_model.fit(X_pred, y)

# Feature importance comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

gb_imp = pd.Series(gb_model.feature_importances_, index=pred_features).sort_values(ascending=True)
gb_imp.plot.barh(ax=axes[0], color='darkorange')
axes[0].set_title('Gradient Boosting Feature Importance')

plot_tree(dt_model, feature_names=pred_features, class_names=['Good','Bad Exit'],
          filled=True, rounded=True, ax=axes[1], fontsize=7)
axes[1].set_title('Decision Tree')

plt.tight_layout()
plt.show()

## 4. Evaluation & Interpretation

In [13]:
# Evaluation focused on recall
y_pred_gb = gb_model.predict(X_pred)
y_prob_gb = gb_model.predict_proba(X_pred)[:, 1]

print('Gradient Boosting Classification Report:')
print(classification_report(y, y_pred_gb, target_names=['Good Outcome', 'Bad Exit']))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(y, y_pred_gb,
    display_labels=['Good','Bad Exit'], ax=axes[0], cmap='Reds')
axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y, y_prob_gb)
auc = roc_auc_score(y, y_prob_gb)
axes[1].plot(fpr, tpr, color='darkorange', label=f'GB (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

prec, rec, thresholds = precision_recall_curve(y, y_prob_gb)
axes[2].plot(rec, prec, color='darkorange')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

print(f'\nBusiness Interpretation:')
print(f'A FALSE NEGATIVE means missing a girl at risk -- the WORST outcome.')
print(f'We optimize for recall even at the cost of some false positives.')
print(f'False positives mean extra monitoring for a girl who may not need it -- acceptable.')

Gradient Boosting Classification Report:
              precision    recall  f1-score   support

Good Outcome       0.91      0.97      0.94        31
    Bad Exit       0.96      0.90      0.93        29

    accuracy                           0.93        60
   macro avg       0.94      0.93      0.93        60
weighted avg       0.94      0.93      0.93        60




Business Interpretation:
A FALSE NEGATIVE means missing a girl at risk -- the WORST outcome.
We optimize for recall even at the cost of some false positives.
False positives mean extra monitoring for a girl who may not need it -- acceptable.


## 5. Causal and Relationship Analysis

### Key Early Warning Indicators

**From the Logistic Regression:**
- **Early incident count** (first 6 months) is the strongest predictor of bad outcomes. Residents with multiple early incidents are significantly more likely to have bad exits.
- **Initial risk level** at intake predicts bad exits, validating the intake assessment tool.
- **Compound trauma score** (number of overlapping trauma categories) increases risk. Girls with multiple forms of abuse/neglect face worse outcomes.
- **Early unfavorable home visits** signal family environments that may not support successful reintegration.
- **Low early emotional improvement** in counseling sessions is a warning sign.

### Causal Defensibility

- Compound trauma score is a static intake feature, not something the organization can change. However, it can inform *intensity* of intervention.
- Early incidents may be both a symptom (trauma manifesting as behavior) and a cause (incidents leading to escalation and transfer).
- Family cooperation is partially modifiable through family engagement programs.

### Actionable Recommendations

1. **Flag residents with compound trauma scores >= 3** for enhanced monitoring from day one.
2. **After any early incident**, immediately review and intensify the intervention plan.
3. **Track emotional trajectory** in counseling -- if no improvement after 3 months, convene a case conference.
4. **Invest in family engagement** programs, especially when early home visits are unfavorable or families are uncooperative.

In [14]:
os.makedirs('models', exist_ok=True)
joblib.dump(gb_model, 'models/bad_exit_gb_model.joblib')
joblib.dump(rf_model, 'models/bad_exit_rf_model.joblib')
joblib.dump(dt_model, 'models/bad_exit_dt_model.joblib')
joblib.dump(pred_features, 'models/bad_exit_features.joblib')
joblib.dump(scaler, 'models/bad_exit_scaler.joblib')
print('Bad exit models saved to models/ directory')

Bad exit models saved to models/ directory


## 6. Deployment Notes

**Model:** Gradient Boosting classifier predicting bad-exit probability from early indicators.

**Integration:**
- `/api/ml/early-warning` endpoint accepts a resident ID and returns:
  - Bad-exit probability
  - Risk tier: Red (>0.7), Yellow (0.3-0.7), Green (<0.3)
  - Top 3 contributing risk factors
- Admin dashboard "Early Warning System" panel:
  - Color-coded risk indicators for all active residents
  - Sorted by risk score (highest first)
  - Drill-down showing which factors are driving the risk
  - Alert when a resident's score crosses into Red zone

**Retraining:** Monthly, incorporating latest counseling, incident, and visit data.

**Threshold Tuning:** The default threshold of 0.5 can be lowered (e.g., 0.3) to increase recall at the cost of more false positives -- a worthwhile tradeoff given the severity of missing a girl at risk.